Задача 7 Rectangles (Shikaku)
Разбейте поле 8 × 8 на прямоугольники со
сторонами по линиям сетки. Каждая клетка
должна принадлежать ровно одному
прямоугольнику. Каждый прямоугольник
должен содержать ровно одну числовую
подсказку - его площадь в клетках должна
равняться этой подсказке.

 .  .  .  6  .  .  .  .

 .  6  .  .  .  .  6  .

 .  .  .  .  6  .  .  .

 .  .  .  .  .  .  .  3

 . 12  .  4  .  .  .  .

 .  .  .  .  .  .  .  .

 .  .  .  .  4 12  .  .

 .  .  5  .  .  .  .  .

Создайте MIP-модель и найдите разбиения с
минимальной и максимальной суммой
периметров прямоугольников.
Подсказка: может иметь смысл заранее
вычислить мультимножество допустимых
прямоугольников (без размещения).
Решения, не основанные на этом подходе,
также приветствуются.


In [53]:
!pip install ortools



Что же от нас требуется, нам необходимо понять какие прямоугольники выбрать

необходимое условие - клетка покрывается 1 раз (но тут важно понимать, что условие покрытия клетки один раз его достаточно и для гарантии попадания покдсказки гарантированно в один прямоугольник и только в один, но если вдруг будет ситация, что сумма подсказок будет менньше чем n x n сетки, то придется передать ограничение на покрытие каждой подсказки, и каждый хотяб 1 должен быть с прямоугольнике, т. е. допускаются ситуации, когда кусок сетки не принадлежит и одному прямугольнику)

будем решать зачаду так: заранее сделаем множество кандидатов, и посмотрим на покрытия, ну и вообще говоря цже можно догадаться, что переменной будет выступать булева перменная и ее смысл барть или не брать этого кандидата для этой подсказки ну и таких переменных p, толчно только же, сколько и подсказок.

Данные у нас следующие:

S — клетки поля, N - длины ширина поля,  |S| = 64;
K — клетки с подсказками, |K| = 10;

Прямоугольник задаётся четвёркой $p = (r_0, c_0, h, w)$ (верхний левый угол, высота, ширина) и несёт множество клеток:

$$S_p = \{ (i, j) : r_0 \le i \le r_0 + h - 1, \ c_0 \le j \le c_0 + w - 1 \}, \quad |S_p| = hw$$

**Кандидатом** называется прямоугольник, удовлетворяющий условиям:

* **(C1)** $1 \le r_0$, $1 \le c_0$, $r_0 + h - 1 \le N$, $c_0 + w - 1 \le N$ (лежит в поле);
* **(C2)** $|S_p \cap K| = 1$ (содержит ровно одну подсказку); обозначим её $k_p$;
* **(C3)** $hw = a_{k_p}$ (площадь равна подсказке).

Множество кандидатов: $R = \{ p : \text{(C1)–(C3)} \}$.


Модель

$$
\begin{cases}
\sum_{p\in R} per_p\, x_p \to \min\ (\max)\\[4pt]
\sum_{p:\ s\in S_p} x_p = 1, & \forall s\in S\\[4pt]
x_p \in \{0,1\}, & \forall p\in R
\end{cases}
$$

Дополнительно можно записать $\sum_{p\in R_k} x_p = 1$ для каждой подсказки k. Как мы выяснили, это то же уравнение, что и для клетки подсказки.


Теперь про модель, ограничение тут такое: x_p это у нас переменная либо 0 либо 1, s это точка на сетке. Мы берем тыкаем в это точку для нее есть некоторое колличество кандидатов, но нам надо выбрать 1 тогда мы раешаем и ставим нашу x_p единицей в этой точке для конкретного прямоугольника, важно понимать, что если его соседняя точка тоже входит в этот прямоугольник, мы не можем выбрать другой прямогольнк на это место, тк сумма уже станет больше единицы, что быдет означать более 1 покрытия, что не допустимо

Для начала пройдем нашу нашу таблицу и найдем подказки, получим массив из подсказок, коотррый представяет собой словарь координаты : значение, считаем от 0

In [54]:
N = 8
GRID = [
    [0, 0, 0, 6, 0, 0, 0, 0],
    [0, 6, 0, 0, 0, 0, 6, 0],
    [0, 0, 0, 0, 6, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 3],
    [0,12, 0, 4, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 4,12, 0, 0],
    [0, 0, 5, 0, 0, 0, 0, 0]
]

clues = {}
for r in range(N):
    for c in range(N):
        if GRID[r][c] != 0:
            clues[(r, c)] = GRID[r][c]

print(clues)

{(0, 3): 6, (1, 1): 6, (1, 6): 6, (2, 4): 6, (3, 7): 3, (4, 1): 12, (4, 3): 4, (6, 4): 4, (6, 5): 12, (7, 2): 5}


Мы уже знаем, что прямоугольник кандитат, если выполянеются условия,

он целиком лежит в поле;

он накрывает клетку подсказки k, внутри нет другой подсказки;

его площадь равна aₖ (по построению, перебираем только делители);

Будем задавать прямугольник четверкой из r, c, h, w, координаты верхнего левого угла, ширина и высота

Тут сразу расскажу про фильтры, сначала просто перебрет делители, которые влезут в нашу сетку, этот как раз все возмонжные ширина и высота.

Далее, задаем диапазон позиций верхнего леого угла, по построению подсказка точно будет лежать в нашем диапазоне, сразу же проверяем границы



In [55]:
rects, rect_clue = [], []

for (cr, cc), area in clues.items():

    #арифметический фильтр 1:
    for h in range(1, N + 1):
        if area % h:      #целочисленное деление ну логично
            continue
        w = area // h
        if w > N:         #ширина влезает ли в поле
            continue
        #cr номер строки
        #задаем диапазон позиций
        for r0 in range(cr - h + 1, cr + 1):         #строка подсказки попадёт в [r0, r0+h-1] если что эта штука просто как длину отрезка показывает она на неотносительно координат
            # раницы поля по вертикали
            if r0 < 0 or r0 + h > N:
                continue
            for c0 in range(cc - w + 1, cc + 1):     #столбец подсказки попадёт в [c0, c0+w-1] тут также
                #границы поля по горизонтали
                if c0 < 0 or c0 + w > N:
                    continue
                #чужая подсказка внутри (проверим входит ли координата друго подсказки в наш диапазон)
                if any(r0 <= r2 < r0 + h and c0 <= c2 < c0 + w
                       for (r2, c2) in clues if (r2, c2) != (cr, cc)):
                    continue
                rects.append((r0, c0, h, w))
                rect_clue.append((cr, cc))

print(rects, '\n', rect_clue)

[(0, 0, 1, 6), (0, 1, 1, 6), (0, 2, 1, 6), (0, 2, 2, 3), (0, 3, 2, 3), (0, 2, 3, 2), (1, 0, 1, 6), (0, 0, 2, 3), (1, 0, 2, 3), (1, 1, 2, 3), (0, 0, 3, 2), (0, 1, 3, 2), (1, 0, 3, 2), (1, 1, 3, 2), (1, 2, 1, 6), (0, 4, 2, 3), (0, 5, 2, 3), (1, 5, 2, 3), (0, 5, 3, 2), (0, 6, 3, 2), (1, 5, 3, 2), (0, 6, 6, 1), (1, 6, 6, 1), (2, 0, 1, 6), (2, 1, 1, 6), (2, 2, 1, 6), (1, 2, 2, 3), (1, 3, 2, 3), (2, 2, 2, 3), (2, 3, 2, 3), (2, 4, 2, 3), (0, 4, 3, 2), (1, 3, 3, 2), (1, 4, 3, 2), (2, 4, 3, 2), (0, 4, 6, 1), (3, 5, 1, 3), (1, 7, 3, 1), (2, 7, 3, 1), (3, 7, 3, 1), (2, 0, 4, 3), (3, 0, 4, 3), (2, 0, 6, 2), (4, 2, 1, 4), (4, 3, 1, 4), (3, 2, 2, 2), (3, 3, 2, 2), (4, 2, 2, 2), (4, 3, 2, 2), (1, 3, 4, 1), (2, 3, 4, 1), (3, 3, 4, 1), (4, 3, 4, 1), (6, 1, 1, 4), (5, 3, 2, 2), (6, 3, 2, 2), (3, 4, 4, 1), (4, 4, 4, 1), (4, 5, 4, 3), (2, 5, 6, 2), (7, 0, 1, 5), (7, 1, 1, 5), (7, 2, 1, 5), (3, 2, 5, 1)] 
 [(0, 3), (0, 3), (0, 3), (0, 3), (0, 3), (0, 3), (1, 1), (1, 1), (1, 1), (1, 1), (1, 1), (1, 1), (1, 

In [56]:
from collections import Counter
cnt = Counter(rect_clue)
for cl in clues:
    print(cl, 'подсказка', clues[cl], '-> кандидатов', cnt[cl])
print('всего:', len(rects))

(0, 3) подсказка 6 -> кандидатов 6
(1, 1) подсказка 6 -> кандидатов 8
(1, 6) подсказка 6 -> кандидатов 9
(2, 4) подсказка 6 -> кандидатов 13
(3, 7) подсказка 3 -> кандидатов 4
(4, 1) подсказка 12 -> кандидатов 3
(4, 3) подсказка 4 -> кандидатов 10
(6, 4) подсказка 4 -> кандидатов 5
(6, 5) подсказка 12 -> кандидатов 2
(7, 2) подсказка 5 -> кандидатов 4
всего: 64


В итоге мы получили 2 массива, первый содержит четверки кондидатов всех возможножных дял каждой подсказки, а второй массив это костыль, который просто хранит в себе координаты подсказки, для который у нас есть четверка, и они равной длины да

In [57]:
P = range(len(rects))

def cells_of(p):
    r0, c0, h, w = rects[p]
    return [(r, c) for r in range(r0, r0 + h) for c in range(c0, c0 + w)]

def perimeter(p):
    return 2 * (rects[p][2] + rects[p][3])

In [58]:
from ortools.linear_solver import pywraplp

In [59]:
def solve(sense):

    solver = pywraplp.Solver.CreateSolver("SCIP")

    #перменные модели: по одной булевой переменной на каждого кандидата.
    #x[p] = 1 прямоугольник p входит в разбиение;  0 не входит.
    x = [solver.IntVar(0, 1, f"x_{p}") for p in P]

    # cover[cell] — список переменных всех кандидатов, накрывающих клетку,

    cover = {(r, c): [] for r in range(N) for c in range(N)}

    for p in P:                       # перебираем кандидатов
        for cell in cells_of(p):      # перебираем клетки, которые p накрывает
            cover[cell].append(x[p])  # переменная кандидата попадает в строки этих клеток

    #каждая клетка покрыта только один раз:
    for vars_ in cover.values():
        solver.Add(sum(vars_) == 1)

    #сумма периметров выбранных прямоугольников
    obj = sum(perimeter(p) * x[p] for p in P)

    # min и max — две ОТДЕЛЬНЫЕ задачи с общим допустимым множеством
    if sense == "min":
        solver.Minimize(obj)
    else:
        solver.Maximize(obj)

    status = solver.Solve()


    if status != pywraplp.Solver.OPTIMAL:
        print("статус:", status)
        return None

    chosen = [p for p in P if x[p].solution_value() > 0.9]


    return solver.Objective().Value(), solver.Objective().BestBound(), chosen

In [60]:
def show(chosen):
    letters = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
    board = [["?"] * N for _ in range(N)]

    for i, p in enumerate(chosen):
        for r, c in cells_of(p):
            board[r][c] = letters[i]

    for r in range(N):
        row = ""
        for c in range(N):
            if (r, c) in clues:
                row += board[r][c].lower() + " "
            else:
                row += board[r][c] + " "
        print(row)

# Сам вывод
for sense in ("min", "max"):
    res = solve(sense)
    if res:
        val, bound, chosen = res

        print(f"\n=== {sense.upper()} ===")
        print(f"Сумма периметров: {val:}\n")
        print(f"Граница: {bound:}\n")
        show(chosen)



=== MIN ===
Сумма периметров: 104.0

Граница: 104.0

B B A a D D C C 
B b A A D D c C 
B B A A d D C C 
F F F G G E E e 
F f F g G I I I 
F F F H H I I I 
F F F H h i I I 
J J j J J I I I 

=== MAX ===
Сумма периметров: 120.0

Граница: 120.0

A A A a A A C C 
B b B B B B c C 
D D D D d D C C 
F F F G H E E e 
F f F g H I I I 
F F F G H I I I 
F F F G h i I I 
J J j J J I I I 
